<a href="https://colab.research.google.com/github/MitraShabani/Merge-Order-Bias/blob/main/merge_order_bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai tiktoken

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = OpenAI()
MODEL_NAME = "gpt-4o-mini"

import tiktoken
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text):
    return len(encoding.encode(text))

print("OpenAI client ready.")

In [ ]:
# Generate text from a prompt
def generate(prompt, max_new_tokens=300):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load and chunk the book
import re

with open("/content/drive/MyDrive/merge-order-bias/Gambler.txt", "r", encoding="utf-8") as f:
    full_text = f.read()

# Simple chapter split — adjust the regex pattern to match the book's actual chapter markers
chapters = re.split(r'\n\s*[IVXLCDM]+\s*\n', full_text)
chapters = [c.strip() for c in chapters if c.strip()]

print(f"Number of chapters found: {len(chapters)}")
for i, ch in enumerate(chapters):
    print(f"Chapter {i+1}: {count_tokens(ch)} tokens")

In [ ]:
# Sub-split any chapter that's too long
MAX_CHUNK_TOKENS = 6000

def split_long_chapter(chapter_text, max_tokens=MAX_CHUNK_TOKENS):
    """Split a chapter into smaller pieces if it exceeds max_tokens, splitting on paragraph breaks."""
    token_count = count_tokens(chapter_text)
    if token_count <= max_tokens:
        return [chapter_text]

    paragraphs = chapter_text.split("\n\n")
    sub_chunks = []
    current = ""
    for para in paragraphs:
        candidate = current + "\n\n" + para if current else para
        if count_tokens(candidate) > max_tokens and current:
            sub_chunks.append(current)
            current = para
        else:
            current = candidate
    if current:
        sub_chunks.append(current)
    return sub_chunks

# Pre-check: show how many pieces each chapter will become
for i, ch in enumerate(chapters):
    pieces = split_long_chapter(ch)
    if len(pieces) > 1:
        print(f"Chapter {i+1} will be split into {len(pieces)} sub-chunks")
    else:
        print(f"Chapter {i+1}: OK, no split needed")

In [ ]:
# Summarize each chapter

def summarize_chapter(chapter_text):
    """Summarize a chapter, sub-splitting first if it's too long, then combining sub-summaries."""
    pieces = split_long_chapter(chapter_text)
    if len(pieces) == 1:
        prompt = f"""Summarize the following book chapter in a concise paragraph, preserving key plot events, characters, and details:

{pieces[0]}

Summary:"""
        return generate(prompt, max_new_tokens=250)
    else:
        # Summarize each piece, then combine those piece-summaries into one chapter summary
        piece_summaries = []
        for piece in pieces:
            prompt = f"""Summarize the following excerpt in a concise paragraph, preserving key plot events, characters, and details:

{piece}

Summary:"""
            piece_summaries.append(generate(prompt, max_new_tokens=200))
        combine_prompt = f"""Combine the following partial summaries of one book chapter into a single concise chapter summary:

{chr(10).join(piece_summaries)}

Combined summary:"""
        return generate(combine_prompt, max_new_tokens=250)


chapter_summaries = []

for i, chapter in enumerate(chapters):
    prompt = f"""Summarize the following book chapter in a concise paragraph, preserving key plot events, characters, and details:

{chapter}

Summary:"""
    summary = summarize_chapter(chapter)
    chapter_summaries.append(summary)

    print(f"--- Chapter {i+1} summary ---")
    print(summary)
    print()

In [ ]:
# Forward Merge Pipeline
"""Merge chapter summaries sequentially"""

def merge_prompt(running_summary, new_chunk_summary, chapters_so_far_count):
    return f"""You are maintaining a running summary of a book, one chapter at a time.

Previous summary (covers chapters 1-{chapters_so_far_count - 1}):
{running_summary}

New chapter content to add (chapter {chapters_so_far_count}):
{new_chunk_summary}

Your task:
1. Write a new summary covering ALL chapters so far (1 through {chapters_so_far_count}), as flowing prose.
2. You MUST weave in specific new plot details from chapter {chapters_so_far_count} — do not skip or gloss over the new chapter.
3. You are free to compress, shorten, or drop minor detail from earlier chapters as needed to fit the new content in — you do not need to preserve every earlier detail verbatim.
4. Do not simply copy the previous summary and append a sentence — genuinely rewrite it as one coherent narrative.

Updated summary:"""

# This lets us later trace which chapters contributed to the final summary
merge_log = []
running_summary = chapter_summaries[0]
merge_log.append({"step": 0, "chapters_included": [1], "summary": running_summary})


for i in range(1, len(chapter_summaries)):
    chapters_so_far_count = i + 1
    prompt = merge_prompt(running_summary, chapter_summaries[i], chapters_so_far_count)
    running_summary = generate(prompt, max_new_tokens= 1200)
    chapters_so_far = list(range(1, chapters_so_far_count + 1))
    merge_log.append({"step": i, "chapters_included": chapters_so_far, "summary": running_summary})

    print(f"--- After merging chapter {i+1} ---")
    print(running_summary)
    print()

final_summary_forward = running_summary
print("=== FINAL FORWARD-MERGED SUMMARY ===")
print(final_summary_forward)

In [ ]:
# Save results
import json

results = {
    "model": MODEL_NAME,
    "merge_order": "forward",
    "num_chapters": len(chapters),
    "chapter_summaries": chapter_summaries,
    "merge_log": merge_log,
    "final_summary": final_summary_forward
}

with open("/content/drive/MyDrive/merge-order-bias/forward_merge_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to forward_merge_results.json")